# Section 1: Project Overview, Problem Definition & Scope

## Objective

Define the ICE train delay prediction project: problem statement, ML tasks, verified data sources, scope, and a reusable configuration saved to disk for all later notebooks.

## What problem does it solve?

ICE trains are critical for long-distance travel in Germany. We ask:

1. **Classification** — Will this ICE stop be delayed?
2. **Regression** — How many minutes late will it be?

Each row = one train stop at one station. Inputs: Deutsche Bahn operational data + Open-Meteo historical weather.

## Methodology

| Source | Role |
|--------|------|
| [piebro/deutsche-bahn-data](https://huggingface.co/datasets/piebro/deutsche-bahn-data) | Schedules, delays, cancellations (~187M rows, monthly Parquet on HF) |
| [Open-Meteo Archive API](https://open-meteo.com/en/docs/historical-weather-api) | Hourly weather via `archive-api.open-meteo.com/v1/archive` |

**Scope:** `train_type == "ICE"` only. Raw data stays on Hugging Face — not in GitHub.

**Known gap:** Railway data has `eva` station IDs but no `latitude`/`longitude`. Geocoding is planned in Notebook 04.

**Alternatives rejected:** all train types (too broad), classification-only (loses delay magnitude), local 54.8 GB download (impractical).

**Reproducibility rule:** Each notebook runs independently. Shared settings are saved to `data/reference/project_config.json` and loaded from file — never assumed from another notebook's memory.

## Expected Output

- Environment check (Python + packages)
- Printed project summary
- `data/reference/project_config.json` created on disk
- Local folders: `data/processed/`, `data/reference/`

## Interpretation

- All packages `OK` → proceed to Section 2
- `MISSING` package → `pip install pandas pyarrow requests`
- 17 DB columns printed → schema verified from Hugging Face (not invented)
- Config file saved → later notebooks can `json.load()` it without re-running Notebook 01

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| 54.8 GB dataset | Stream monthly Parquet from HF |
| No coordinates in DB data | Geocoding in Notebook 04 |
| Schema docs vs live data | Use `train_number`/`line_number` (not `train_name`) |

## Summary

Problem, tasks, sources, and scope are defined. Configuration is persisted to disk for reproducible, independent notebook execution.

## Next Step

**Section 2:** Research questions, delay definition, success metrics, and testable hypotheses.

---

### Notebook 01 — Progress Notes

**Key Takeaways**
- Two tasks: binary delay classification + delay regression (`delay_in_min`)
- ICE-only filter; one row = one stop
- Weather merge requires station geocoding first
- Config saved to `data/reference/project_config.json`

**What Comes Next**
- Section 2: formal research questions and evaluation criteria
- Section 3+: data loading, cleaning, merge (later notebooks load saved files)

In [1]:
# =============================================================================
# Notebook 01 | Section 1: Project Overview, Problem Definition & Scope
# =============================================================================
# Self-contained: runs top-to-bottom with no prior notebook state.
# Saves project_config.json so later notebooks load settings from disk.
# =============================================================================

from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

# =============================================================================
# 1. PATHS — single source of truth for this project
# =============================================================================
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"

for folder in [DATA_DIR, PROCESSED_DIR, REFERENCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 2. PROJECT CONFIGURATION (verified against Hugging Face Dataset Viewer)
# =============================================================================
PROJECT_CONFIG = {
    "project": {
        "title": (
            "ICE Train Delay Prediction Using Railway Operational Data "
            "and Historical Weather Data"
        ),
        "author": "Your Name",           # TODO: replace
        "institution": "Your University",  # TODO: replace
        "notebook": "01_Project_Definition_and_Data_Understanding",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    },
    "scope": {
        "train_type_filter": "ICE",
        "observation_unit": "one train stop at one station (one row)",
        "exclude_from_git": "full raw Deutsche Bahn dataset (54.8 GB)",
    },
    "ml_tasks": {
        "classification": {
            "question": "Will this ICE train stop be delayed?",
            "target": "is_delayed",
            "target_source": "derived from delay_in_min in Notebook 06",
        },
        "regression": {
            "question": "How many minutes late will this ICE train stop be?",
            "target": "delay_in_min",
            "target_source": "column in Deutsche Bahn dataset",
        },
    },
    "data_sources": {
        "deutsche_bahn": {
            "hf_dataset_id": "piebro/deutsche-bahn-data",
            "hf_url": "https://huggingface.co/datasets/piebro/deutsche-bahn-data",
            "monthly_parquet_pattern": "monthly_processed_data/data-{year_month}.parquet",
            "approx_rows": 187_311_724,
            "approx_size_gb": 54.8,
            "license": "CC BY 4.0 (Deutsche Bahn attribution required)",
        },
        "open_meteo": {
            "archive_url": "https://archive-api.open-meteo.com/v1/archive",
            "docs_url": "https://open-meteo.com/en/docs/historical-weather-api",
            "required_params": [
                "latitude", "longitude", "start_date", "end_date", "hourly"
            ],
            "candidate_hourly_variables": [
                "temperature_2m",
                "precipitation",
                "rain",
                "snowfall",
                "windspeed_10m",
                "windgusts_10m",
                "weather_code",
                "visibility",
            ],
        },
    },
    "db_verified_columns": [
        {"name": "station_name",           "dtype": "string",    "role": "station label (nullable)"},
        {"name": "xml_station_name",       "dtype": "string",    "role": "API/XML station name"},
        {"name": "eva",                    "dtype": "string",    "role": "EVA station ID — geocoding key"},
        {"name": "train_number",           "dtype": "string",    "role": "train service number"},
        {"name": "line_number",            "dtype": "string",    "role": "line ID (nullable)"},
        {"name": "final_destination_station","dtype": "string",  "role": "route endpoint"},
        {"name": "delay_in_min",           "dtype": "int32",     "role": "regression target"},
        {"name": "time",                   "dtype": "timestamp", "role": "record reference time"},
        {"name": "is_canceled",            "dtype": "bool",      "role": "cancellation flag"},
        {"name": "train_type",             "dtype": "string",    "role": "filter: ICE only"},
        {"name": "train_line_ride_id",     "dtype": "string",    "role": "unique ride ID"},
        {"name": "train_line_station_num", "dtype": "int32",     "role": "stop order on route"},
        {"name": "arrival_planned_time",   "dtype": "timestamp", "role": "scheduled arrival"},
        {"name": "arrival_change_time",    "dtype": "timestamp", "role": "actual/revised arrival"},
        {"name": "departure_planned_time", "dtype": "timestamp", "role": "scheduled departure"},
        {"name": "departure_change_time",  "dtype": "timestamp", "role": "actual/revised departure"},
        {"name": "id",                     "dtype": "string",    "role": "unique row ID"},
    ],
    "db_missing_columns": [
        {"name": "latitude",  "solution": "geocode eva/xml_station_name in Notebook 04"},
        {"name": "longitude", "solution": "geocode eva/xml_station_name in Notebook 04"},
    ],
    "paths": {
        "project_root": str(PROJECT_ROOT),
        "data_dir": str(DATA_DIR),
        "processed_dir": str(PROCESSED_DIR),
        "reference_dir": str(REFERENCE_DIR),
        "config_path": str(CONFIG_PATH),
    },
    "notebook_roadmap": [
        "01 — Project Definition & Data Understanding",
        "02 — Data Acquisition & Initial Inspection",
        "03 — Data Cleaning & Standardization",
        "04 — Data Integration (Merge Deutsche Bahn + Weather)",
        "05 — Exploratory Data Analysis",
        "06 — Feature Engineering",
        "07 — Baseline Machine Learning Models",
        "08 — Advanced Machine Learning Models",
        "09 — Model Evaluation & Hyperparameter Tuning",
        "10 — Final Prediction Pipeline & Conclusion",
    ],
    "reproducibility": {
        "rule": "Each notebook is self-contained; load shared config and data from disk.",
        "config_file": str(CONFIG_PATH),
    },
}

# =============================================================================
# 3. HELPER — load config in any later notebook (copy this pattern)
# =============================================================================
def load_project_config(path: Path = CONFIG_PATH) -> dict:
    """
  Load project configuration from JSON.
  Use in Notebooks 02–10 instead of relying on Notebook 01 kernel state.

  Example (later notebook):
      config = load_project_config()
      ice_filter = config["scope"]["train_type_filter"]
  """
    if not path.exists():
        raise FileNotFoundError(
            f"Config not found: {path}\n"
            "Run Notebook 01 Section 1 first to create project_config.json."
        )
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# 4. ENVIRONMENT CHECK
# =============================================================================
def check_package(import_name: str, pip_name: str | None = None) -> str:
    """Return 'OK' if importable, else install hint."""
    try:
        __import__(import_name)
        return "OK"
    except ImportError:
        return f"MISSING — pip install {pip_name or import_name}"


REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "requests": "requests",
}

package_status = {name: check_package(name, pip) for name, pip in REQUIRED_PACKAGES.items()}

# =============================================================================
# 5. SAVE CONFIG TO DISK (later notebooks load this file)
# =============================================================================
with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(PROJECT_CONFIG, f, indent=2, ensure_ascii=False)

# Verify round-trip: load from file (proves later notebooks can do the same)
loaded_config = load_project_config()

# =============================================================================
# 6. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print(loaded_config["project"]["title"])
print(sep)
print(f"Author      : {loaded_config['project']['author']}")
print(f"Institution : {loaded_config['project']['institution']}")
print(f"Notebook    : {loaded_config['project']['notebook']}")
print(f"Section     : {loaded_config['project']['section']}")
print()

print("--- SCOPE ---")
print(f"Train filter : {loaded_config['scope']['train_type_filter']}")
print(f"Row unit     : {loaded_config['scope']['observation_unit']}")
print()

print("--- ML TASKS ---")
for task, info in loaded_config["ml_tasks"].items():
    print(f"  {task}: {info['question']}")
    print(f"    target → {info['target']}")
print()

print("--- DATA SOURCES ---")
db = loaded_config["data_sources"]["deutsche_bahn"]
om = loaded_config["data_sources"]["open_meteo"]
print(f"  DB  : {db['hf_url']}")
print(f"  HF  : {db['hf_dataset_id']}  (~{db['approx_rows']:,} rows)")
print(f"  Meteo: {om['archive_url']}")
print()

print(f"--- VERIFIED DB COLUMNS ({len(loaded_config['db_verified_columns'])}) ---")
for i, col in enumerate(loaded_config["db_verified_columns"], 1):
    print(f"  {i:2d}. {col['name']:<28} {col['dtype']}")
print()

print("--- MISSING IN DB (external fix required) ---")
for col in loaded_config["db_missing_columns"]:
    print(f"  - {col['name']}: {col['solution']}")
print()

print("--- PATHS ---")
for key, val in loaded_config["paths"].items():
    print(f"  {key}: {val}")
print()

print("--- ENVIRONMENT ---")
print(f"  Python : {sys.version.split()[0]}")
for pkg, status in package_status.items():
    print(f"  {pkg:10s}: {status}")
print()

print(f"--- CONFIG SAVED ---")
print(f"  {CONFIG_PATH}")
print(f"  Reload test: OK ({len(loaded_config['db_verified_columns'])} columns loaded from file)")
print()

missing = [p for p, s in package_status.items() if not s.startswith("OK")]
if missing:
    print("ACTION REQUIRED:")
    print("  pip install " + " ".join(missing))
else:
    print("Environment OK. Config persisted. Ready for Section 2.")

print(sep)

ICE Train Delay Prediction Using Railway Operational Data and Historical Weather Data
Author      : Your Name
Institution : Your University
Notebook    : 01_Project_Definition_and_Data_Understanding
Section     : Section 1

--- SCOPE ---
Train filter : ICE
Row unit     : one train stop at one station (one row)

--- ML TASKS ---
  classification: Will this ICE train stop be delayed?
    target → is_delayed
  regression: How many minutes late will this ICE train stop be?
    target → delay_in_min

--- DATA SOURCES ---
  DB  : https://huggingface.co/datasets/piebro/deutsche-bahn-data
  HF  : piebro/deutsche-bahn-data  (~187,311,724 rows)
  Meteo: https://archive-api.open-meteo.com/v1/archive

--- VERIFIED DB COLUMNS (17) ---
   1. station_name                 string
   2. xml_station_name             string
   3. eva                          string
   4. train_number                 string
   5. line_number                  string
   6. final_destination_station    string
   7. delay_in_m

# Section 2: Research Questions, Delay Definition & Success Criteria

## Objective

Formulate testable research questions, define how "delayed" is operationalized for classification, specify evaluation metrics for both ML tasks, and save this framework to disk for later notebooks.

## Why this step is important

Metrics and targets must be defined **before** modeling. Without a fixed delay threshold and success criteria, results cannot be compared fairly or defended in a viva.

## What problem does it solve?

It translates the business problem (*"Will the train be late?"*) into measurable ML objectives with clear hypotheses linking weather to delays.

## Methodology

**Research questions**
- **RQ1:** Can operational features alone predict ICE delays?
- **RQ2:** Does adding weather data improve delay prediction?
- **RQ3:** Which weather variables correlate most with delay severity?

**Delay definition (classification)**  
`is_delayed = 1` if `delay_in_min >= DELAY_THRESHOLD_MINUTES`.  
Default: **6 minutes** — aligns with Deutsche Bahn's common "significant delay" reporting band. Final threshold will be validated in Notebook 05 (EDA).

**Metrics**

| Task | Primary metrics | Why |
|------|-----------------|-----|
| Classification | F1, ROC-AUC, confusion matrix | Handles class imbalance better than accuracy alone |
| Regression | MAE, RMSE | MAE = avg error in minutes; RMSE penalizes large errors |

**Hypotheses (to test in Notebooks 05–09)**
- H1: Higher precipitation increases delay probability.
- H2: Extreme wind (`windgusts_10m`) increases delay magnitude.
- H3: Weather features improve model performance vs operational-only baseline.

**Alternatives:** 5-min threshold (simpler, noisier); accuracy-only evaluation (misleading if most trains are on time); random train/test split (rejected — temporal split used later).

**Persistence:** Framework saved to `data/reference/research_framework.json` — loaded from file in all later notebooks.

## Expected Output

- Loaded `project_config.json`
- Printed research questions, threshold, metrics, hypotheses
- New file: `data/reference/research_framework.json`

## Interpretation

- Config loads without error → Section 1 was run and saved correctly
- Threshold printed as 6 min → default until EDA confirms or adjusts
- JSON saved → Notebooks 06–09 load the same definitions without re-running Notebook 01

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| Class imbalance (most stops on time) | Use F1/ROC-AUC, not accuracy alone |
| Negative `delay_in_min` (early arrival) | Treat as on-time in classification; keep raw value for regression |
| Threshold choice is subjective | Document DB convention; validate in EDA |

## Summary

Research questions, delay rule, metrics, and hypotheses are defined and saved to disk. Modeling notebooks will load this file — not notebook memory.

## Next Step

**Section 3:** Deep dive into Deutsche Bahn data dictionary — column meanings, null patterns, and ICE-filter implications.

---

**Key Takeaways**
- Two-task evaluation plan: F1/ROC-AUC (classification), MAE/RMSE (regression)
- Delay threshold = 6 min (configurable, EDA-validated later)
- Weather value tested via with/without-weather model comparison (RQ2)

**What Comes Next**
- Section 3: Deutsche Bahn schema and column-level documentation

In [2]:
# =============================================================================
# Notebook 01 | Section 2: Research Questions, Delay Definition & Success Criteria
# =============================================================================
# Self-contained: loads project_config.json from disk (no prior kernel state).
# Saves research_framework.json for Notebooks 05–09.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

# =============================================================================
# 1. PATHS
# =============================================================================
PROJECT_ROOT = Path.cwd()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
RESEARCH_FRAMEWORK_PATH = REFERENCE_DIR / "research_framework.json"

REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 2. LOAD PROJECT CONFIG (from file — not from Notebook 01 memory)
# =============================================================================
def load_project_config(path: Path = CONFIG_PATH) -> dict:
    """Load shared project configuration saved in Notebook 01."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Run Notebook 01 Section 1 first to create project_config.json."
        )
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_research_framework(path: Path = RESEARCH_FRAMEWORK_PATH) -> dict:
    """Load research framework — use this in Notebooks 05–09."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Run Notebook 01 Section 2 first to create research_framework.json."
        )
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


config = load_project_config()

# =============================================================================
# 3. RESEARCH FRAMEWORK DEFINITION
# =============================================================================

# Delay threshold for binary classification
# DB often reports "significant delays" from ~6 minutes in passenger info.
# We treat early arrivals (negative delay) as on-time (is_delayed = 0).
DELAY_THRESHOLD_MINUTES = 6

RESEARCH_FRAMEWORK = {
    "metadata": {
        "notebook": config["project"]["notebook"],
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "depends_on": str(CONFIG_PATH),
    },
    "research_questions": {
        "RQ1": {
            "question": "Can operational features alone predict ICE train stop delays?",
            "answered_in": ["Notebook 07", "Notebook 09"],
            "method": "Train baseline models using only railway features.",
        },
        "RQ2": {
            "question": "Does adding historical weather data improve delay prediction?",
            "answered_in": ["Notebook 07", "Notebook 09"],
            "method": "Compare models with vs without weather features (ablation).",
        },
        "RQ3": {
            "question": "Which weather variables are most associated with ICE delays?",
            "answered_in": ["Notebook 05", "Notebook 09"],
            "method": "EDA correlations + feature importance / SHAP in final models.",
        },
    },
    "classification": {
        "target_column": "is_delayed",
        "derivation": {
            "source_column": "delay_in_min",
            "rule": f"is_delayed = 1 if delay_in_min >= {DELAY_THRESHOLD_MINUTES}, else 0",
            "early_arrival_rule": "delay_in_min < 0 treated as on-time (is_delayed = 0)",
            "canceled_rule": "is_canceled == True rows excluded in Notebook 03 (documented there)",
        },
        "delay_threshold_minutes": DELAY_THRESHOLD_MINUTES,
        "threshold_justification": (
            "6 minutes aligns with Deutsche Bahn significant-delay reporting. "
            "Final value validated in Notebook 05 EDA."
        ),
        "primary_metrics": ["f1_score", "roc_auc", "confusion_matrix"],
        "secondary_metrics": ["precision", "recall", "accuracy"],
        "metric_rationale": (
            "ICE stops are mostly on-time (class imbalance). "
            "F1 and ROC-AUC are more informative than accuracy alone."
        ),
    },
    "regression": {
        "target_column": "delay_in_min",
        "target_filter": "Non-canceled stops; delay can be negative (early) or positive (late).",
        "primary_metrics": ["mae", "rmse"],
        "secondary_metrics": ["r2_score", "median_absolute_error"],
        "metric_rationale": (
            "MAE is interpretable in minutes. "
            "RMSE penalizes large prediction errors (severe delays)."
        ),
    },
    "hypotheses": {
        "H1": {
            "statement": "Higher hourly precipitation increases ICE delay probability.",
            "type": "classification",
            "test": "Compare delay rate across precipitation bins in EDA; "
                    "check weather feature importance in classifier.",
        },
        "H2": {
            "statement": "Higher wind gusts increase expected delay magnitude.",
            "type": "regression",
            "test": "Correlation/regression EDA; wind feature coefficient/importance.",
        },
        "H3": {
            "statement": "Models with weather features outperform operational-only baselines.",
            "type": "both",
            "test": "Ablation study: same algorithm, with vs without weather columns.",
        },
    },
    "evaluation_protocol": {
        "train_test_split": "time-based (no random shuffle) — implemented in Notebook 07",
        "baseline_requirement": "Must beat naive baseline before using advanced models",
        "naive_baselines": {
            "classification": "always predict majority class (on-time)",
            "regression": "always predict mean delay_in_min",
        },
        "weather_ablation": {
            "description": "Train identical model twice: with and without weather features",
            "purpose": "Directly answers RQ2",
        },
    },
    "alternatives_considered": {
        "delay_threshold_5_min": "Rejected as default — more noise from minor timing differences",
        "delay_threshold_15_min": "Rejected — too few positive cases for classification",
        "accuracy_only": "Rejected — misleading under class imbalance",
        "random_train_test_split": "Rejected — causes temporal data leakage",
    },
}

# =============================================================================
# 4. HELPER — derive is_delayed (used again in Notebook 06; defined here for clarity)
# =============================================================================
def derive_is_delayed(delay_in_min: int, threshold: int = DELAY_THRESHOLD_MINUTES) -> int:
    """
    Convert delay_in_min to binary is_delayed label.

    Parameters
    ----------
    delay_in_min : int
        Delay in minutes from Deutsche Bahn data (negative = early).
    threshold : int
        Minutes at or above which a stop counts as delayed.

    Returns
    -------
    int : 1 if delayed, 0 if on-time or early
    """
    if delay_in_min < 0:
        return 0
    return int(delay_in_min >= threshold)


# Quick sanity check on the derivation logic
_assert_examples = [
    (derive_is_delayed(-1), 0, "early arrival"),
    (derive_is_delayed(0), 0, "on time"),
    (derive_is_delayed(5), 0, "below threshold"),
    (derive_is_delayed(6), 1, "at threshold"),
    (derive_is_delayed(45), 1, "late"),
]
for result, expected, label in _assert_examples:
    assert result == expected, f"Failed for {label}: got {result}, expected {expected}"

# =============================================================================
# 5. SAVE TO DISK
# =============================================================================
with RESEARCH_FRAMEWORK_PATH.open("w", encoding="utf-8") as f:
    json.dump(RESEARCH_FRAMEWORK, f, indent=2, ensure_ascii=False)

# Round-trip verification
framework = load_research_framework()

# =============================================================================
# 6. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 2: Research Questions, Delay Definition & Success Criteria")
print(sep)
print(f"Project : {config['project']['title']}")
print(f"Config  : loaded from {CONFIG_PATH}")
print()

print("--- RESEARCH QUESTIONS ---")
for rq_id, rq in framework["research_questions"].items():
    print(f"  {rq_id}: {rq['question']}")
print()

print("--- CLASSIFICATION ---")
cls = framework["classification"]
print(f"  Target     : {cls['target_column']}")
print(f"  Threshold  : {cls['delay_threshold_minutes']} minutes")
print(f"  Rule       : {cls['derivation']['rule']}")
print(f"  Metrics    : {', '.join(cls['primary_metrics'])}")
print()

print("--- REGRESSION ---")
reg = framework["regression"]
print(f"  Target     : {reg['target_column']}")
print(f"  Metrics    : {', '.join(reg['primary_metrics'])}")
print()

print("--- HYPOTHESES ---")
for h_id, h in framework["hypotheses"].items():
    print(f"  {h_id}: {h['statement']}")
print()

print("--- DERIVATION SANITY CHECK ---")
examples = [(-1, "early"), (0, "on-time"), (5, "5 min"), (6, "6 min"), (30, "30 min")]
for delay, label in examples:
    print(f"  delay_in_min={delay:3d} ({label:8s}) → is_delayed = {derive_is_delayed(delay)}")
print()

print(f"--- SAVED ---")
print(f"  {RESEARCH_FRAMEWORK_PATH}")
print(f"  Reload test: OK ({len(framework['hypotheses'])} hypotheses loaded from file)")
print()
print("Ready for Section 3: Deutsche Bahn Data Dictionary.")
print(sep)

Section 2: Research Questions, Delay Definition & Success Criteria
Project : ICE Train Delay Prediction Using Railway Operational Data and Historical Weather Data
Config  : loaded from c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\project_config.json

--- RESEARCH QUESTIONS ---
  RQ1: Can operational features alone predict ICE train stop delays?
  RQ2: Does adding historical weather data improve delay prediction?
  RQ3: Which weather variables are most associated with ICE delays?

--- CLASSIFICATION ---
  Target     : is_delayed
  Threshold  : 6 minutes
  Rule       : is_delayed = 1 if delay_in_min >= 6, else 0
  Metrics    : f1_score, roc_auc, confusion_matrix

--- REGRESSION ---
  Target     : delay_in_min
  Metrics    : mae, rmse

--- HYPOTHESES ---
  H1: Higher hourly precipitation increases ICE delay probability.
  H2: Higher wind gusts increase expected delay magnitude.
  H3: Models with weather features outperform operational-only baselines.

--- DERIVATION SANI

# Section 3: Deutsche Bahn Data Dictionary & ICE Scope

## Objective

Document every verified column in the Deutsche Bahn dataset, inspect a real sample from Hugging Face, profile null patterns, and save a reusable data dictionary to disk.

## Why this step is important

You cannot clean, merge, or model data you do not understand. A data dictionary prevents invented columns and gives you viva-ready explanations for each field.

## What problem does it solve?

It answers: *What does each column mean, which columns are targets/keys, and what happens when we filter to ICE only?*

## Methodology

1. Load `project_config.json` from disk.
2. Fetch a **small sample** (100 rows) via the Hugging Face Dataset Viewer API — not the full 54.8 GB dataset.
3. Filter `train_type == "ICE"` and profile dtypes + null rates.
4. Document all **17 verified columns** with ML role (key, feature, target, metadata).
5. Save `data/reference/db_data_dictionary.json` for Notebooks 02–04.

**Note:** Live schema uses `train_number` + `line_number`, not `train_name` from the dataset card.

## Expected Output

- Sample load status (API or documented fallback)
- ICE row count in sample
- Per-column dtype and null %
- Example ICE row printed
- Saved `db_data_dictionary.json`

## Interpretation

- **High null % on `station_name`** → use `xml_station_name` or `eva` as backup identifiers.
- **`line_number` nullable** → many ICE rows have `null` line numbers; do not drop rows solely for this.
- **`delay_in_min` can be negative** → early arrival; handled in Section 2 classification rule.
- **No lat/lon columns** → confirms geocoding is required before weather merge (Notebook 04).

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| 187M rows — cannot load all | HF Dataset Viewer API sample (100 rows) |
| API timeout possible | Fallback verified sample embedded in code |
| Mixed train types in sample | Filter `train_type == "ICE"` after load |

## Summary

All 17 Deutsche Bahn columns are documented with ML roles. A real sample is profiled and the dictionary is persisted for downstream notebooks.

## Next Step

**Section 4:** Open-Meteo API structure, required parameters, and weather variable selection.

---

**Key Takeaways**
- `eva` = station key; `delay_in_min` = regression target; `train_type` filters ICE
- Timestamp columns need standardization before merge (Notebook 03)
- `id` and `train_line_ride_id` are identifiers — not features

**What Comes Next**
- Section 4: Open-Meteo historical weather API documentation and sample request

In [3]:
# =============================================================================
# Notebook 01 | Section 3: Deutsche Bahn Data Dictionary & ICE Scope
# =============================================================================
# Self-contained: loads config from disk; fetches small HF sample via API.
# Saves db_data_dictionary.json for Notebooks 02–04.
# Does NOT download the full 54.8 GB dataset.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests

# =============================================================================
# 1. PATHS & CONFIG
# =============================================================================
PROJECT_ROOT = Path.cwd()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
DB_DICTIONARY_PATH = REFERENCE_DIR / "db_data_dictionary.json"

REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

HF_DATASET_ID = "piebro/deutsche-bahn-data"
HF_ROWS_API = "https://datasets-server.huggingface.co/rows"
SAMPLE_LENGTH = 100          # small sample only
REQUEST_TIMEOUT_SEC = 60
ICE_FILTER = "ICE"


def load_project_config(path: Path = CONFIG_PATH) -> dict:
    """Load project config from disk (created in Section 1)."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\nRun Notebook 01 Section 1 first."
        )
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_db_data_dictionary(path: Path = DB_DICTIONARY_PATH) -> dict:
    """Load DB data dictionary — use in Notebooks 02–04."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\nRun Notebook 01 Section 3 first."
        )
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


config = load_project_config()
VERIFIED_COLUMNS = [c["name"] for c in config["db_verified_columns"]]

# =============================================================================
# 2. COLUMN DICTIONARY (verified schema — Hugging Face Dataset Viewer)
# =============================================================================
DB_DATA_DICTIONARY: dict[str, dict[str, Any]] = {
    "station_name": {
        "dtype": "string",
        "nullable": True,
        "description": "Human-readable station name. Can be null; prefer xml_station_name or eva.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "xml_station_name": {
        "dtype": "string",
        "nullable": False,
        "description": "Station name as returned by Deutsche Bahn XML/API.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "eva": {
        "dtype": "string",
        "nullable": False,
        "description": "EVA number — official German station identifier (8-digit style, e.g. 08000207).",
        "ml_role": "key",
        "merge_key": True,
        "note": "Primary station key for geocoding before weather merge.",
    },
    "train_number": {
        "dtype": "string",
        "nullable": False,
        "description": "Train service number (e.g. 292, 618, 892 for ICE services).",
        "ml_role": "feature",
        "merge_key": False,
    },
    "line_number": {
        "dtype": "string",
        "nullable": True,
        "description": "Line identifier. Often null for ICE long-distance services.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "final_destination_station": {
        "dtype": "string",
        "nullable": False,
        "description": "Terminal station of the train route.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "delay_in_min": {
        "dtype": "int32",
        "nullable": False,
        "description": "Delay in minutes. Negative = early; 0 = on time; positive = late.",
        "ml_role": "target",
        "merge_key": False,
        "note": "Regression target. Source for is_delayed in classification.",
    },
    "time": {
        "dtype": "timestamp[ns]",
        "nullable": False,
        "description": "Reference timestamp for the record.",
        "ml_role": "feature",
        "merge_key": False,
        "note": "Standardize to 24h datetime in Notebook 03.",
    },
    "is_canceled": {
        "dtype": "bool",
        "nullable": False,
        "description": "True if this train stop was canceled.",
        "ml_role": "filter",
        "merge_key": False,
        "note": "Typically excluded from modeling in Notebook 03.",
    },
    "train_type": {
        "dtype": "string",
        "nullable": False,
        "description": "Vehicle category (ICE, IC, RE, S, Bus, etc.).",
        "ml_role": "filter",
        "merge_key": False,
        "note": f"Keep only rows where train_type == '{ICE_FILTER}'.",
    },
    "train_line_ride_id": {
        "dtype": "string",
        "nullable": False,
        "description": "Unique identifier for one complete train journey.",
        "ml_role": "identifier",
        "merge_key": False,
        "note": "Useful for grouping stops; not a direct weather merge key.",
    },
    "train_line_station_num": {
        "dtype": "int32",
        "nullable": False,
        "description": "Stop sequence number on the route (1 = first stop).",
        "ml_role": "feature",
        "merge_key": False,
    },
    "arrival_planned_time": {
        "dtype": "timestamp[ns]",
        "nullable": True,
        "description": "Scheduled arrival at this station.",
        "ml_role": "feature",
        "merge_key": False,
        "note": "Nullable at first stop / departure-only rows.",
    },
    "arrival_change_time": {
        "dtype": "timestamp[ns]",
        "nullable": True,
        "description": "Actual or revised arrival time.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "departure_planned_time": {
        "dtype": "timestamp[ns]",
        "nullable": True,
        "description": "Scheduled departure from this station.",
        "ml_role": "feature",
        "merge_key": True,
        "note": "Primary candidate for hour-level weather merge key (Notebook 04).",
    },
    "departure_change_time": {
        "dtype": "timestamp[ns]",
        "nullable": True,
        "description": "Actual or revised departure time.",
        "ml_role": "feature",
        "merge_key": False,
    },
    "id": {
        "dtype": "string",
        "nullable": False,
        "description": "Unique identifier for one train stop record.",
        "ml_role": "identifier",
        "merge_key": False,
        "note": "Row-level primary key; do not use as ML feature.",
    },
}

# Sanity: dictionary must match verified config column list
_dict_cols = set(DB_DATA_DICTIONARY.keys())
_config_cols = set(VERIFIED_COLUMNS)
assert _dict_cols == _config_cols, (
    f"Column mismatch!\n  dictionary only: {_dict_cols - _config_cols}\n"
    f"  config only: {_config_cols - _dict_cols}"
)

# =============================================================================
# 3. FETCH SMALL SAMPLE FROM HUGGING FACE (Dataset Viewer API)
# =============================================================================
def fetch_hf_sample(
    dataset_id: str,
    length: int = SAMPLE_LENGTH,
    offset: int = 0,
    timeout: int = REQUEST_TIMEOUT_SEC,
) -> tuple[pd.DataFrame, str]:
    """
    Fetch rows from Hugging Face Dataset Viewer API.
    Returns (DataFrame, source_label).
    """
    params = {
        "dataset": dataset_id,
        "config": "default",
        "split": "train",
        "offset": offset,
        "length": length,
    }
    response = requests.get(HF_ROWS_API, params=params, timeout=timeout)
    response.raise_for_status()
    payload = response.json()

    rows = [item["row"] for item in payload.get("rows", [])]
    if not rows:
        raise ValueError("API returned zero rows.")

    df = pd.DataFrame(rows)
    source = f"HF Dataset Viewer API (offset={offset}, length={length})"
    return df, source


# Verified fallback: real ICE rows from Hugging Face Dataset Viewer (July 2024)
# Used only if API is unavailable — values are NOT invented.
FALLBACK_SAMPLE_ROWS = [
    {
        "station_name": "Braunschweig Hbf",
        "xml_station_name": "Braunschweig Hbf",
        "eva": "08000049",
        "train_number": "292",
        "line_number": None,
        "final_destination_station": "Berlin Ostbahnhof",
        "delay_in_min": 0,
        "time": "2024-07-01T00:01:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "-8314087786935469967",
        "train_line_station_num": 14,
        "arrival_planned_time": "2024-06-30T23:59:00",
        "arrival_change_time": "2024-06-30T23:57:00",
        "departure_planned_time": "2024-07-01T00:01:00",
        "departure_change_time": "2024-07-01T00:01:00",
        "id": "-8314087786935469967-2406301659-14",
    },
    {
        "station_name": "Mainz Hbf",
        "xml_station_name": "Mainz Hbf",
        "eva": "08000240",
        "train_number": "920",
        "line_number": None,
        "final_destination_station": "Hamburg-Altona",
        "delay_in_min": 0,
        "time": "2024-07-01T00:01:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "8503777394879306018",
        "train_line_station_num": 3,
        "arrival_planned_time": "2024-06-30T23:59:00",
        "arrival_change_time": "2024-06-30T23:59:00",
        "departure_planned_time": "2024-07-01T00:01:00",
        "departure_change_time": "2024-07-01T00:01:00",
        "id": "8503777394879306018-2406302324-3",
    },
    {
        "station_name": "München Hbf",
        "xml_station_name": "München Hbf",
        "eva": "08000261",
        "train_number": "618",
        "line_number": None,
        "final_destination_station": "Kiel Hbf",
        "delay_in_min": 1,
        "time": "2024-07-01T00:02:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "2686007473625185344",
        "train_line_station_num": 1,
        "arrival_planned_time": None,
        "arrival_change_time": None,
        "departure_planned_time": "2024-07-01T00:01:00",
        "departure_change_time": "2024-07-01T00:02:00",
        "id": "2686007473625185344-2407010001-1",
    },
    {
        "station_name": "Essen Hbf",
        "xml_station_name": "Essen Hbf",
        "eva": "08000098",
        "train_number": "892",
        "line_number": None,
        "final_destination_station": "Köln Hbf",
        "delay_in_min": 1,
        "time": "2024-07-01T00:03:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "-309830498189753415",
        "train_line_station_num": 13,
        "arrival_planned_time": "2024-07-01T00:00:00",
        "arrival_change_time": "2024-07-01T00:01:00",
        "departure_planned_time": "2024-07-01T00:02:00",
        "departure_change_time": "2024-07-01T00:03:00",
        "id": "-309830498189753415-2406301934-13",
    },
    {
        "station_name": "Hamburg-Altona",
        "xml_station_name": "Hamburg-Altona",
        "eva": "08002553",
        "train_number": "730",
        "line_number": None,
        "final_destination_station": "Hamburg-Altona",
        "delay_in_min": 0,
        "time": "2024-07-01T00:04:00",
        "is_canceled": False,
        "train_type": "ICE",
        "train_line_ride_id": "-3889563661905709281",
        "train_line_station_num": 13,
        "arrival_planned_time": "2024-07-01T00:04:00",
        "arrival_change_time": "2024-07-01T00:04:00",
        "departure_planned_time": None,
        "departure_change_time": None,
        "id": "-3889563661905709281-2406301936-13",
    },
]

try:
    sample_df, sample_source = fetch_hf_sample(HF_DATASET_ID, length=SAMPLE_LENGTH)
except Exception as api_error:
    print(f"WARNING: HF API unavailable ({api_error}). Using verified fallback sample.")
    sample_df = pd.DataFrame(FALLBACK_SAMPLE_ROWS)
    sample_source = "Verified fallback (Hugging Face Dataset Viewer, July 2024)"

# Ensure expected columns exist (guard against API schema changes)
missing_cols = [c for c in VERIFIED_COLUMNS if c not in sample_df.columns]
if missing_cols:
    raise ValueError(
        f"Unexpected schema — missing columns: {missing_cols}\n"
        "Re-inspect https://huggingface.co/datasets/piebro/deutsche-bahn-data"
    )

# Reorder columns to match verified schema
sample_df = sample_df[VERIFIED_COLUMNS]

# =============================================================================
# 4. ICE FILTER & PROFILING
# =============================================================================
ice_df = sample_df[sample_df["train_type"] == ICE_FILTER].copy()

# Dtype conversion for timestamp columns (still strings from JSON API)
timestamp_cols = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]
for col in timestamp_cols:
    sample_df[col] = pd.to_datetime(sample_df[col], errors="coerce")
    if not ice_df.empty:
        ice_df[col] = pd.to_datetime(ice_df[col], errors="coerce")

def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Return per-column dtype and null percentage."""
    profile = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "null_count": [int(df[c].isna().sum()) for c in df.columns],
        "null_pct": [round(100 * df[c].isna().mean(), 1) for c in df.columns],
        "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
    })
    return profile.sort_values("null_pct", ascending=False).reset_index(drop=True)


full_profile = profile_dataframe(sample_df)
ice_profile = profile_dataframe(ice_df) if not ice_df.empty else pd.DataFrame()

# Train type distribution in sample
train_type_counts = (
    sample_df["train_type"]
    .value_counts(dropna=False)
    .rename_axis("train_type")
    .reset_index(name="count")
)

# Delay stats for ICE rows in sample
ice_delay_stats = {}
if not ice_df.empty:
    ice_delay_stats = {
        "min": int(ice_df["delay_in_min"].min()),
        "max": int(ice_df["delay_in_min"].max()),
        "mean": round(float(ice_df["delay_in_min"].mean()), 2),
        "negative_count": int((ice_df["delay_in_min"] < 0).sum()),
        "zero_count": int((ice_df["delay_in_min"] == 0).sum()),
        "positive_count": int((ice_df["delay_in_min"] > 0).sum()),
    }

# =============================================================================
# 5. BUILD & SAVE DATA DICTIONARY ARTIFACT
# =============================================================================
dictionary_artifact = {
    "metadata": {
        "notebook": config["project"]["notebook"],
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "hf_dataset_id": HF_DATASET_ID,
        "hf_url": config["data_sources"]["deutsche_bahn"]["hf_url"],
        "sample_source": sample_source,
        "sample_rows_total": int(len(sample_df)),
        "sample_rows_ice": int(len(ice_df)),
        "ice_filter": ICE_FILTER,
    },
    "columns": DB_DATA_DICTIONARY,
    "column_order": VERIFIED_COLUMNS,
    "sample_profile": {
        "train_type_distribution": train_type_counts.to_dict(orient="records"),
        "full_null_profile": full_profile.to_dict(orient="records"),
        "ice_null_profile": ice_profile.to_dict(orient="records") if not ice_profile.empty else [],
        "ice_delay_stats": ice_delay_stats,
    },
    "ice_scope_notes": [
        "One row = one train stop at one station.",
        f"Filter: train_type == '{ICE_FILTER}'.",
        "line_number is frequently null for ICE — expected, not a data error.",
        "departure_planned_time is the primary temporal anchor for weather merge.",
        "No latitude/longitude — geocode eva in Notebook 04.",
    ],
    "columns_not_for_modeling": ["id", "train_line_ride_id"],
    "proposed_merge_keys": {
        "station": "eva → latitude, longitude (after geocoding)",
        "time": "departure_planned_time rounded to hour → Open-Meteo hourly.time",
    },
}

with DB_DICTIONARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(dictionary_artifact, f, indent=2, ensure_ascii=False)

loaded_dictionary = load_db_data_dictionary()

# =============================================================================
# 6. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 3: Deutsche Bahn Data Dictionary & ICE Scope")
print(sep)
print(f"Dataset  : {HF_DATASET_ID}")
print(f"Sample   : {sample_source}")
print(f"Rows     : {len(sample_df)} total | {len(ice_df)} ICE")
print()

print("--- TRAIN TYPE DISTRIBUTION (sample) ---")
print(train_type_counts.to_string(index=False))
print()

print("--- COLUMN DICTIONARY (17 columns) ---")
for col in VERIFIED_COLUMNS:
    meta = DB_DATA_DICTIONARY[col]
    print(f"  {col:<28} | {meta['dtype']:<15} | {meta['ml_role']:<10} | {meta['description'][:50]}...")
print()

print("--- NULL PROFILE (all sample rows, sorted by null %) ---")
print(full_profile.to_string(index=False))
print()

if ice_delay_stats:
    print("--- ICE DELAY STATS (sample) ---")
    for k, v in ice_delay_stats.items():
        print(f"  {k}: {v}")
    print()

if not ice_df.empty:
    print("--- EXAMPLE ICE ROW ---")
    example = ice_df.iloc[0]
    for col in VERIFIED_COLUMNS:
        print(f"  {col:<28}: {example[col]}")
    print()

print(f"--- SAVED ---")
print(f"  {DB_DICTIONARY_PATH}")
print(f"  Reload test: OK ({len(loaded_dictionary['columns'])} columns in file)")
print()
print("Ready for Section 4: Open-Meteo API documentation.")
print(sep)

Section 3: Deutsche Bahn Data Dictionary & ICE Scope
Dataset  : piebro/deutsche-bahn-data
Sample   : HF Dataset Viewer API (offset=0, length=100)
Rows     : 100 total | 5 ICE

--- TRAIN TYPE DISTRIBUTION (sample) ---
train_type  count
         S     57
        RB     14
        RE     10
       ICE      5
       Bus      4
       MEX      3
       HLB      2
       FEX      2
       STB      1
       SBB      1
       BRB      1

--- COLUMN DICTIONARY (17 columns) ---
  station_name                 | string          | feature    | Human-readable station name. Can be null; prefer x...
  xml_station_name             | string          | feature    | Station name as returned by Deutsche Bahn XML/API....
  eva                          | string          | key        | EVA number — official German station identifier (8...
  train_number                 | string          | feature    | Train service number (e.g. 292, 618, 892 for ICE s...
  line_number                  | string          | feat

# Section 4: Open-Meteo Historical Weather API — Structure & Sample Request

## Objective

Document the Open-Meteo Archive API response structure, request a real weather sample for a German station location, parse it into a DataFrame, and save the weather data dictionary to disk.

## Why this step is important

Weather data has a different schema than railway data. We must verify the **real JSON response** before designing merge keys in Notebook 04.

## What problem does it solve?

Railway rows have `eva` station IDs but **no coordinates**. Open-Meteo requires `latitude` + `longitude`. This section confirms what the API returns and which hourly fields we will join to ICE stops.

## Methodology

1. Load `project_config.json` and `db_data_dictionary.json` from disk.
2. Call `https://archive-api.open-meteo.com/v1/archive` with verified parameters.
3. Use **Berlin area coordinates** (52.525, 13.369) as a demo — same date as our DB sample (`2024-07-01`).
4. Request hourly variables: `temperature_2m`, `precipitation`, `rain`, `snowfall`, `windspeed_10m`, `windgusts_10m`, `weather_code`, `visibility`.
5. Parse `hourly.time` + weather arrays into a DataFrame.
6. Save `weather_data_dictionary.json` and cache sample response to `data/reference/open_meteo_sample_response.json`.

**Merge logic (planned Notebook 04):** `eva → (lat, lon)` + `departure_planned_time` rounded to hour → `hourly.time`.

## Expected Output

- HTTP 200 from Open-Meteo
- 24 hourly rows for one day (one row per hour)
- JSON response cached on disk
- `weather_data_dictionary.json` saved

## Interpretation

- **`hourly.time`** format `YYYY-MM-DDTHH:00` in requested timezone (`Europe/Berlin`) — this becomes the weather-side merge key.
- **Returned lat/lon may differ slightly** from request (API snaps to weather grid) — normal behavior.
- **`precipitation`** = preceding hour sum in mm; **`temperature_2m`** = instant value in °C.
- **No station name in response** — confirms geocoding step is mandatory.

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| Railway data has no coordinates | Geocode `eva` in Notebook 04 |
| Hourly weather vs minute-level trains | Round `departure_planned_time` to hour |
| API rate limits on large merges | Cache responses per station-month locally |

## Summary

Open-Meteo API structure is verified with a real request. Hourly weather fields and time format are documented and saved for the merge step.

## Next Step

**Section 5:** Merge strategy design — keys, join type, validation checks, and end-to-end data pipeline diagram.

---

**Key Takeaways**
- Weather API needs `latitude`, `longitude`, `start_date`, `end_date`, `hourly`
- Response is JSON with parallel arrays under `hourly`
- Timezone must be `Europe/Berlin` for German rail data

**What Comes Next**
- Section 5: formal merge plan + Notebook 01 closing summary

In [4]:
# =============================================================================
# Notebook 01 | Section 4: Open-Meteo Historical Weather API
# =============================================================================
# Self-contained: loads config from disk; calls real Open-Meteo Archive API.
# Saves weather_data_dictionary.json + cached sample response JSON.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests

# =============================================================================
# 1. PATHS & LOAD CONFIG FROM DISK
# =============================================================================
PROJECT_ROOT = Path.cwd()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
DB_DICTIONARY_PATH = REFERENCE_DIR / "db_data_dictionary.json"
WEATHER_DICTIONARY_PATH = REFERENCE_DIR / "weather_data_dictionary.json"
SAMPLE_RESPONSE_PATH = REFERENCE_DIR / "open_meteo_sample_response.json"

REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
OPEN_METEO_DOCS_URL = "https://open-meteo.com/en/docs/historical-weather-api"
REQUEST_TIMEOUT_SEC = 60

# Demo location: Berlin area (used only to verify API structure in this section)
# In Notebook 04, each eva station gets its own coordinates via geocoding.
DEMO_LATITUDE = 52.525
DEMO_LONGITUDE = 13.369
DEMO_START_DATE = "2024-07-01"   # matches DB sample date from Section 3
DEMO_END_DATE = "2024-07-01"
DEMO_TIMEZONE = "Europe/Berlin"

# Hourly variables selected for this project (documented in Open-Meteo API)
HOURLY_VARIABLES = [
    "temperature_2m",    # °C — air temperature at 2 m
    "precipitation",     # mm — total precip in preceding hour
    "rain",              # mm — liquid rain in preceding hour
    "snowfall",          # cm — snow in preceding hour
    "windspeed_10m",     # km/h — wind speed at 10 m
    "windgusts_10m",     # km/h — wind gusts at 10 m
    "weather_code",      # WMO code — categorical weather condition
    "visibility",        # m — horizontal visibility
]


def load_json(path: Path) -> dict:
    """Load a JSON file; raise clear error if missing."""
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_weather_data_dictionary(path: Path = WEATHER_DICTIONARY_PATH) -> dict:
    """Load weather dictionary — use in Notebooks 03–04."""
    return load_json(path)


config = load_json(CONFIG_PATH)
db_dictionary = load_json(DB_DICTIONARY_PATH)

# =============================================================================
# 2. WEATHER VARIABLE DICTIONARY (from Open-Meteo API documentation)
# =============================================================================
WEATHER_VARIABLE_DOCS: dict[str, dict[str, Any]] = {
    "temperature_2m": {
        "unit": "°C",
        "valid_time": "instant",
        "description": "Air temperature at 2 metres above ground.",
        "relevance": "Extreme heat/cold may affect infrastructure and operations.",
    },
    "precipitation": {
        "unit": "mm",
        "valid_time": "preceding_hour_sum",
        "description": "Total precipitation (rain, showers, snow) in the preceding hour.",
        "relevance": "Wet tracks, reduced adhesion, visibility issues.",
    },
    "rain": {
        "unit": "mm",
        "valid_time": "preceding_hour_sum",
        "description": "Liquid rain in the preceding hour.",
        "relevance": "Component of total precipitation affecting operations.",
    },
    "snowfall": {
        "unit": "cm",
        "valid_time": "preceding_hour_sum",
        "description": "Snowfall in the preceding hour.",
        "relevance": "Winter disruption risk on ICE corridors.",
    },
    "windspeed_10m": {
        "unit": "km/h",
        "valid_time": "instant",
        "description": "Wind speed at 10 metres above ground.",
        "relevance": "Strong winds affect high-speed running.",
    },
    "windgusts_10m": {
        "unit": "km/h",
        "valid_time": "instant",
        "description": "Maximum wind gusts at 10 metres.",
        "relevance": "Captures short severe weather events.",
    },
    "weather_code": {
        "unit": "wmo_code",
        "valid_time": "instant",
        "description": "WMO weather interpretation code (categorical).",
        "relevance": "Encodes fog, thunderstorm, snow conditions in one field.",
    },
    "visibility": {
        "unit": "m",
        "valid_time": "instant",
        "description": "Horizontal visibility distance.",
        "relevance": "Fog and low visibility can cause speed restrictions.",
    },
}

# =============================================================================
# 3. API REQUEST FUNCTION
# =============================================================================
def fetch_open_meteo_archive(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    hourly_variables: list[str],
    timezone: str = "Europe/Berlin",
    timeout: int = REQUEST_TIMEOUT_SEC,
) -> dict:
    """
    Call Open-Meteo Historical Weather Archive API.

    Parameters
    ----------
    latitude, longitude : float
        WGS84 coordinates (from geocoding in Notebook 04).
    start_date, end_date : str
        Format YYYY-MM-DD.
    hourly_variables : list[str]
        Weather variables to request (comma-joined in API call).
    timezone : str
        Timezone for hourly.time timestamps.

    Returns
    -------
    dict : Raw JSON response from the API.
    """
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": timezone,
    }

    response = requests.get(
        OPEN_METEO_ARCHIVE_URL,
        params=params,
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()


def parse_hourly_response(api_response: dict) -> pd.DataFrame:
    """
    Convert Open-Meteo hourly JSON into a flat DataFrame.

    The API returns parallel arrays:
        hourly.time, hourly.temperature_2m, hourly.precipitation, ...
    Each index across arrays = one hour.
    """
    if "hourly" not in api_response:
        raise ValueError("API response missing 'hourly' key.")

    hourly = api_response["hourly"]
    if "time" not in hourly:
        raise ValueError("API response missing 'hourly.time'.")

    df = pd.DataFrame(hourly)

    # Parse time to datetime (24-hour format, timezone-aware string from API)
    df["time"] = pd.to_datetime(df["time"])

    # Add metadata columns useful for merge debugging
    df["latitude"] = api_response.get("latitude")
    df["longitude"] = api_response.get("longitude")
    df["timezone"] = api_response.get("timezone")

    return df


# =============================================================================
# 4. EXECUTE REAL API CALL
# =============================================================================
print("Calling Open-Meteo Archive API...")
api_response = fetch_open_meteo_archive(
    latitude=DEMO_LATITUDE,
    longitude=DEMO_LONGITUDE,
    start_date=DEMO_START_DATE,
    end_date=DEMO_END_DATE,
    hourly_variables=HOURLY_VARIABLES,
    timezone=DEMO_TIMEZONE,
)

# Cache raw response for reproducibility (later notebooks can reload without API call)
with SAMPLE_RESPONSE_PATH.open("w", encoding="utf-8") as f:
    json.dump(api_response, f, indent=2)

# Parse to DataFrame
weather_df = parse_hourly_response(api_response)

# Verify all requested variables arrived in response
response_hourly_keys = set(api_response["hourly"].keys())
expected_keys = set(HOURLY_VARIABLES) | {"time"}
missing_vars = expected_keys - response_hourly_keys
if missing_vars:
    raise ValueError(f"API response missing hourly variables: {missing_vars}")

# =============================================================================
# 5. BUILD & SAVE WEATHER DATA DICTIONARY
# =============================================================================
weather_dictionary = {
    "metadata": {
        "notebook": config["project"]["notebook"],
        "section": "Section 4",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "api_url": OPEN_METEO_ARCHIVE_URL,
        "docs_url": OPEN_METEO_DOCS_URL,
        "demo_request": {
            "latitude": DEMO_LATITUDE,
            "longitude": DEMO_LONGITUDE,
            "start_date": DEMO_START_DATE,
            "end_date": DEMO_END_DATE,
            "timezone": DEMO_TIMEZONE,
            "hourly_variables": HOURLY_VARIABLES,
            "note": "Demo only — Notebook 04 uses per-station geocoded coordinates.",
        },
        "sample_response_path": str(SAMPLE_RESPONSE_PATH),
        "sample_rows": int(len(weather_df)),
    },
    "required_request_parameters": {
        "latitude": "WGS84 latitude — obtained by geocoding eva in Notebook 04",
        "longitude": "WGS84 longitude — obtained by geocoding eva in Notebook 04",
        "start_date": "YYYY-MM-DD — match railway date range being processed",
        "end_date": "YYYY-MM-DD — match railway date range being processed",
        "hourly": "Comma-separated list of weather variables",
        "timezone": "Europe/Berlin — aligns with German rail timestamps",
    },
    "response_top_level_keys": {
        "latitude": "Grid-snapped latitude returned by API (may differ from request)",
        "longitude": "Grid-snapped longitude returned by API",
        "timezone": "Timezone used for hourly.time",
        "hourly_units": "Unit for each hourly variable",
        "hourly": "Object containing parallel time-series arrays",
    },
    "hourly_variables": WEATHER_VARIABLE_DOCS,
    "hourly_time_format": {
        "example": str(weather_df["time"].iloc[0]),
        "pattern": "YYYY-MM-DDTHH:00 in requested timezone",
        "merge_rule": (
            "Round railway departure_planned_time to nearest hour, "
            "then join on (latitude, longitude, hour)"
        ),
    },
    "merge_plan": {
        "railway_keys": db_dictionary["proposed_merge_keys"],
        "weather_keys": {
            "location": "latitude + longitude (after geocoding eva)",
            "time": "hourly.time (hour-rounded)",
        },
        "join_type": "left join (keep all ICE railway rows; weather may be null if missing)",
        "implemented_in": "Notebook 04",
    },
    "verified_response_units": api_response.get("hourly_units", {}),
    "columns_not_in_weather_api": [
        "station_name", "eva", "train_number", "delay_in_min",
        "note: weather API is location+time only — no railway fields",
    ],
}

with WEATHER_DICTIONARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(weather_dictionary, f, indent=2, ensure_ascii=False)

loaded_weather_dict = load_weather_data_dictionary()

# =============================================================================
# 6. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 4: Open-Meteo Historical Weather API — Structure & Sample Request")
print(sep)
print(f"API URL    : {OPEN_METEO_ARCHIVE_URL}")
print(f"Demo coords: ({DEMO_LATITUDE}, {DEMO_LONGITUDE}) — Berlin area")
print(f"Date range : {DEMO_START_DATE} to {DEMO_END_DATE}")
print(f"Timezone   : {DEMO_TIMEZONE}")
print()

print("--- API RESPONSE METADATA ---")
print(f"  Returned latitude  : {api_response.get('latitude')}")
print(f"  Returned longitude : {api_response.get('longitude')}")
print(f"  Timezone           : {api_response.get('timezone')}")
print(f"  Elevation (m)      : {api_response.get('elevation')}")
print()

print("--- HOURLY UNITS (verified from API) ---")
for var, unit in api_response.get("hourly_units", {}).items():
    print(f"  {var}: {unit}")
print()

print(f"--- PARSED DATAFRAME ({len(weather_df)} rows = 1 day × 24 hours) ---")
display_cols = ["time"] + HOURLY_VARIABLES
print(weather_df[display_cols].head(3).to_string(index=False))
print("  ...")
print(weather_df[display_cols].tail(1).to_string(index=False))
print()

print("--- HOURLY VARIABLES FOR THIS PROJECT ---")
for var in HOURLY_VARIABLES:
    doc = WEATHER_VARIABLE_DOCS[var]
    print(f"  {var:<18} {doc['unit']:<10} {doc['description'][:55]}")
print()

print("--- MERGE PLAN (Notebook 04) ---")
print(f"  Railway : {db_dictionary['proposed_merge_keys']}")
print(f"  Weather : {weather_dictionary['merge_plan']['weather_keys']}")
print(f"  Join    : {weather_dictionary['merge_plan']['join_type']}")
print()

print("--- SAVED FILES ---")
print(f"  {WEATHER_DICTIONARY_PATH}")
print(f"  {SAMPLE_RESPONSE_PATH}")
print(f"  Reload test: OK ({len(loaded_weather_dict['hourly_variables'])} variables documented)")
print()
print("Ready for Section 5: Merge Strategy & Pipeline Design.")
print(sep)

Calling Open-Meteo Archive API...
Section 4: Open-Meteo Historical Weather API — Structure & Sample Request
API URL    : https://archive-api.open-meteo.com/v1/archive
Demo coords: (52.525, 13.369) — Berlin area
Date range : 2024-07-01 to 2024-07-01
Timezone   : Europe/Berlin

--- API RESPONSE METADATA ---
  Returned latitude  : 52.54833
  Returned longitude : 13.407822
  Timezone           : Europe/Berlin
  Elevation (m)      : 35.0

--- HOURLY UNITS (verified from API) ---
  time: iso8601
  temperature_2m: °C
  precipitation: mm
  rain: mm
  snowfall: cm
  windspeed_10m: km/h
  windgusts_10m: km/h
  weather_code: wmo code
  visibility: undefined

--- PARSED DATAFRAME (24 rows = 1 day × 24 hours) ---
               time  temperature_2m  precipitation  rain  snowfall  windspeed_10m  windgusts_10m  weather_code visibility
2024-07-01 00:00:00            17.0            0.0   0.0       0.0           12.6           21.6             3       None
2024-07-01 01:00:00            16.9           

# Section 5: Merge Strategy, Pipeline Design & Notebook Summary

## Objective

Define the formal merge strategy between Deutsche Bahn ICE data and Open-Meteo weather, document validation checks, save the integration plan to disk, and close Notebook 01.

## Why this step is important

A wrong join (wrong station, wrong hour, wrong timezone) silently corrupts every model downstream. The merge plan must be written **before** Notebook 04 executes it.

## What problem does it solve?

It answers: *How exactly will railway rows and weather rows be linked, and how will we know the merge succeeded?*

## Methodology

**Merge keys (left = railway, right = weather):**

| Railway field | Transform | Weather field |
|---------------|-----------|---------------|
| `eva` | geocode → `latitude`, `longitude` | `latitude`, `longitude` |
| `departure_planned_time` | round to hour, `Europe/Berlin` | `hourly.time` |

**Join type:** Left join — keep all ICE stops; weather null if lookup fails.

**Validation checks (Notebook 04):** merge rate ≥ 90%, no duplicate row keys, timezone consistent, no future weather on past labels.

**Pipeline:** Load DB → filter ICE → clean timestamps → geocode stations → fetch/cache weather → join → validate → save `ice_weather_merged.parquet`.

All prior artifacts loaded from `data/reference/*.json` — not from notebook memory.

## Expected Output

- Printed merge strategy summary
- Saved `data/reference/merge_strategy.json`
- Notebook 01 checklist complete

## Interpretation

- Merge keys use **planned departure** (not actual) → avoids target leakage
- Hour rounding → weather is hourly; trains are minute-precise
- Left join → we see how many stops lack weather (merge rate metric)
- All 5 JSON config files on disk → Notebook 02 can start independently

## Challenges Faced

| Challenge | Resolution |
|-----------|------------|
| No lat/lon in railway data | Geocode `eva` in Notebook 04 |
| 54.8 GB dataset | Monthly Parquet streaming from Hugging Face |
| Class imbalance | F1/ROC-AUC defined in Section 2 |
| Raw sample ≠ ICE-only | ICE filter applied at load time in Notebook 02 |

## Summary

Notebook 01 is complete. Problem, research framework, both data dictionaries, weather API structure, and merge strategy are defined and saved to disk.

## Next Step

**Notebook 02 — Section 1:** Load Deutsche Bahn monthly Parquet, filter ICE, save `ice_raw_sample.parquet`.

---

### Notebook 01 — Closing

**Key Takeaways**
- Two ML tasks: classification (`is_delayed`) + regression (`delay_in_min`)
- 17 verified DB columns; 8 weather variables; no invented fields
- Merge: `eva → coords` + `departure_planned_time` (hour) → `hourly.time`
- Every notebook loads config from `data/reference/` — fully reproducible

**What Comes Next**
- Notebook 02: acquire and inspect real ICE data (monthly Parquet from Hugging Face)

In [5]:
# =============================================================================
# Notebook 01 | Section 5: Merge Strategy, Pipeline Design & Notebook Summary
# =============================================================================
# Self-contained: loads ALL reference JSON files from disk.
# Saves merge_strategy.json — the blueprint for Notebook 04.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# =============================================================================
# 1. PATHS
# =============================================================================
PROJECT_ROOT = Path.cwd()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"

CONFIG_PATH = REFERENCE_DIR / "project_config.json"
RESEARCH_PATH = REFERENCE_DIR / "research_framework.json"
DB_DICT_PATH = REFERENCE_DIR / "db_data_dictionary.json"
WEATHER_DICT_PATH = REFERENCE_DIR / "weather_data_dictionary.json"
MERGE_STRATEGY_PATH = REFERENCE_DIR / "merge_strategy.json"

REFERENCE_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path: Path) -> dict:
    """Load JSON from disk with a clear error if the file is missing."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\n"
            f"Run the earlier section that creates this file first."
        )
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_merge_strategy(path: Path = MERGE_STRATEGY_PATH) -> dict:
    """Load merge strategy — use in Notebook 04."""
    return load_json(path)


# =============================================================================
# 2. LOAD ALL NOTEBOOK 01 ARTIFACTS (from disk, not kernel memory)
# =============================================================================
config = load_json(CONFIG_PATH)
research = load_json(RESEARCH_PATH)
db_dict = load_json(DB_DICT_PATH)
weather_dict = load_json(WEATHER_DICT_PATH)

# Verify all prior sections were completed
required_files = {
    "Section 1 — project_config": CONFIG_PATH,
    "Section 2 — research_framework": RESEARCH_PATH,
    "Section 3 — db_data_dictionary": DB_DICT_PATH,
    "Section 4 — weather_data_dictionary": WEATHER_DICT_PATH,
}
for label, path in required_files.items():
  status = "OK" if path.exists() else "MISSING"
  print(f"  [{status}] {label}: {path.name}")

missing = [label for label, path in required_files.items() if not path.exists()]
if missing:
  raise FileNotFoundError(
      f"Complete these sections first: {missing}"
  )

# =============================================================================
# 3. MERGE STRATEGY DEFINITION
# =============================================================================
MERGE_STRATEGY: dict[str, Any] = {
    "metadata": {
        "notebook": config["project"]["notebook"],
        "section": "Section 5",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "implemented_in": "Notebook 04 — Data Integration",
        "depends_on": [str(p) for p in required_files.values()],
    },

    # ---- Step-by-step pipeline (Notebooks 02–04) ----
    "pipeline_steps": [
        {
            "step": 1,
            "notebook": "02",
            "action": "Load monthly Parquet from Hugging Face",
            "input": "hf://datasets/piebro/deutsche-bahn-data/monthly_processed_data/data-YYYY-MM.parquet",
            "output": "data/processed/ice_raw_YYYY-MM.parquet",
        },
        {
            "step": 2,
            "notebook": "02",
            "action": "Filter train_type == 'ICE'",
            "input": "raw monthly parquet",
            "output": "ICE-only subset saved to disk",
        },
        {
            "step": 3,
            "notebook": "03",
            "action": "Standardize all timestamps to 24h datetime, timezone Europe/Berlin",
            "columns": [
                "time",
                "arrival_planned_time",
                "arrival_change_time",
                "departure_planned_time",
                "departure_change_time",
            ],
            "output": "data/processed/ice_cleaned_YYYY-MM.parquet",
        },
        {
            "step": 4,
            "notebook": "04",
            "action": "Geocode unique eva station IDs to latitude/longitude",
            "input": "unique eva values from ICE data",
            "output": "data/reference/station_coordinates.json",
            "api": "Open-Meteo Geocoding API or Nominatim (documented in Notebook 04)",
        },
        {
            "step": 5,
            "notebook": "04",
            "action": "Fetch hourly weather per station-month from Open-Meteo Archive",
            "input": "station coordinates + date range",
            "output": "data/processed/weather_YYYY-MM.parquet",
            "cache_rule": "Cache per station-month to avoid repeated API calls",
        },
        {
            "step": 6,
            "notebook": "04",
            "action": "Create merge keys on both sides",
            "railway_key": "eva + departure_planned_hour",
            "weather_key": "latitude + longitude + hourly.time",
        },
        {
            "step": 7,
            "notebook": "04",
            "action": "Left join weather onto railway rows",
            "output": "data/processed/ice_weather_merged_YYYY-MM.parquet",
        },
        {
            "step": 8,
            "notebook": "04",
            "action": "Run validation checks (see validation_checks below)",
            "output": "data/reference/merge_validation_report.json",
        },
    ],

    # ---- Formal merge key specification ----
    "merge_keys": {
        "railway_side": {
            "station_key": {
                "source_column": "eva",
                "transform": "lookup eva in station_coordinates.json → latitude, longitude",
                "why": "Railway data has eva but no coordinates; weather API needs lat/lon",
            },
            "time_key": {
                "source_column": "departure_planned_time",
                "transform": "floor to hour in Europe/Berlin timezone",
                "output_column": "departure_planned_hour",
                "why": (
                    "Open-Meteo returns hourly data. "
                    "Planned departure (not actual) avoids target leakage."
                ),
            },
        },
        "weather_side": {
            "location_key": {
                "columns": ["latitude", "longitude"],
                "note": "API may snap coords to weather grid — use returned values",
            },
            "time_key": {
                "column": "time",
                "format": "YYYY-MM-DDTHH:00 in Europe/Berlin",
                "source": "hourly.time from Open-Meteo response",
            },
        },
        "join_condition": (
            "railway.latitude  == weather.latitude AND "
            "railway.longitude == weather.longitude AND "
            "railway.departure_planned_hour == weather.time"
        ),
        "join_type": "left",
        "join_type_rationale": (
            "Keep all ICE railway rows. "
            "Rows without weather match are flagged, not dropped."
        ),
    },

    # ---- Why each key was chosen ----
    "merge_key_justification": {
        "eva_not_station_name": (
            "eva is a standardized numeric ID; station_name can be null or inconsistent"
        ),
        "departure_planned_not_actual": (
            "Using arrival_change_time or departure_change_time would leak "
            "the delay outcome into the features — the actual time IS the delay"
        ),
        "hour_not_minute": (
            "Weather variables are hourly aggregates; minute-level join "
            "would require interpolation not supported by the API"
        ),
        "left_not_inner": (
            "Inner join would silently drop stops with missing weather, "
            "hiding data quality problems"
        ),
    },

    # ---- Validation checks for Notebook 04 ----
    "validation_checks": [
        {
            "check": "merge_rate",
            "rule": "≥ 90% of ICE rows have non-null weather values",
            "action_if_fail": "Investigate missing geocoding or date range gaps",
        },
        {
            "check": "no_duplicate_row_keys",
            "rule": "id column must be unique after merge",
            "action_if_fail": "Inspect join — weather side may have duplicate hours",
        },
        {
            "check": "timezone_consistency",
            "rule": "All timestamps in Europe/Berlin; no mixed UTC/local",
            "action_if_fail": "Re-run Notebook 03 standardization",
        },
        {
            "check": "no_target_leakage",
            "rule": "Features must not include arrival_change_time, departure_change_time, or delay_in_min",
            "action_if_fail": "Remove leaky columns before modeling in Notebook 06",
        },
        {
            "check": "weather_null_pattern",
            "rule": "Null weather should be random (geocoding gaps), not systematic by route",
            "action_if_fail": "Fix geocoding for affected stations",
        },
        {
            "check": "row_count_preserved",
            "rule": "Row count after left join == row count before join",
            "action_if_fail": "Join created duplicates — inspect weather cache",
        },
    ],

    # ---- Columns after merge ----
    "expected_merged_columns": {
        "railway_columns": config["db_verified_columns"],
        "weather_columns": list(weather_dict["hourly_variables"].keys()),
        "derived_columns": [
            "departure_planned_hour",
            "latitude",
            "longitude",
            "is_delayed",
        ],
    },

    # ---- Leakage prevention ----
    "columns_excluded_from_features": [
        "id",
        "train_line_ride_id",
        "delay_in_min",
        "arrival_change_time",
        "departure_change_time",
        "is_delayed",
        "note: delay_in_min is regression TARGET; change times leak the outcome",
    ],

    # ---- Output files for downstream notebooks ----
    "output_files": {
        "merged_data_pattern": "data/processed/ice_weather_merged_YYYY-MM.parquet",
        "station_coords": "data/reference/station_coordinates.json",
        "validation_report": "data/reference/merge_validation_report.json",
    },
}

# =============================================================================
# 4. NOTEBOOK 01 COMPLETION CHECKLIST
# =============================================================================
NOTEBOOK_01_CHECKLIST = {
    "sections_completed": {
        "Section 1": CONFIG_PATH.exists(),
        "Section 2": RESEARCH_PATH.exists(),
        "Section 3": DB_DICT_PATH.exists(),
        "Section 4": WEATHER_DICT_PATH.exists(),
        "Section 5": True,  # this section
    },
    "artifacts_on_disk": {
        "project_config.json": CONFIG_PATH.exists(),
        "research_framework.json": RESEARCH_PATH.exists(),
        "db_data_dictionary.json": DB_DICT_PATH.exists(),
        "weather_data_dictionary.json": WEATHER_DICT_PATH.exists(),
        "open_meteo_sample_response.json": (
            REFERENCE_DIR / "open_meteo_sample_response.json"
        ).exists(),
        "merge_strategy.json": True,  # saved below
    },
    "ready_for_notebook_02": all([
        CONFIG_PATH.exists(),
        RESEARCH_PATH.exists(),
        DB_DICT_PATH.exists(),
        WEATHER_DICT_PATH.exists(),
    ]),
}

# =============================================================================
# 5. SAVE MERGE STRATEGY TO DISK
# =============================================================================
with MERGE_STRATEGY_PATH.open("w", encoding="utf-8") as f:
    json.dump(MERGE_STRATEGY, f, indent=2, ensure_ascii=False)

loaded_strategy = load_merge_strategy()

# =============================================================================
# 6. PRINT NOTEBOOK 01 SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 5: Merge Strategy, Pipeline Design & Notebook Summary")
print(sep)
print(f"Project: {config['project']['title']}")
print()

print("--- MERGE KEYS ---")
rk = loaded_strategy["merge_keys"]["railway_side"]
wk = loaded_strategy["merge_keys"]["weather_side"]
print(f"  Railway station : {rk['station_key']['source_column']} → geocode → lat/lon")
print(f"  Railway time      : {rk['time_key']['source_column']} → floor to hour")
print(f"  Weather location  : {wk['location_key']['columns']}")
print(f"  Weather time      : {wk['time_key']['column']} ({wk['time_key']['format']})")
print(f"  Join type         : {loaded_strategy['merge_keys']['join_type']}")
print()

print("--- PIPELINE (Notebooks 02–04) ---")
for step in loaded_strategy["pipeline_steps"]:
    print(f"  Step {step['step']} [NB{step['notebook']}]: {step['action']}")
print()

print("--- VALIDATION CHECKS (Notebook 04) ---")
for check in loaded_strategy["validation_checks"]:
    print(f"  • {check['check']}: {check['rule']}")
print()

print("--- NOTEBOOK 01 ARTIFACTS ON DISK ---")
all_artifacts = [
    CONFIG_PATH,
    RESEARCH_PATH,
    DB_DICT_PATH,
    WEATHER_DICT_PATH,
    MERGE_STRATEGY_PATH,
    REFERENCE_DIR / "open_meteo_sample_response.json",
]
for path in all_artifacts:
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {path.name}")
print()

print("--- NOTEBOOK 01 CHECKLIST ---")
for section, done in NOTEBOOK_01_CHECKLIST["sections_completed"].items():
    print(f"  [{'✓' if done else '✗'}] {section}")
print()

ready = NOTEBOOK_01_CHECKLIST["ready_for_notebook_02"]
print(f"--- STATUS ---")
print(f"  Notebook 01 complete : YES")
print(f"  Ready for Notebook 02: {'YES' if ready else 'NO — run missing sections'}")
print(f"  Merge strategy saved : {MERGE_STRATEGY_PATH}")
print()
print("=" * 72)
print("NOTEBOOK 01 COMPLETE")
print("=" * 72)
print()
print("Key Takeaways:")
print("  • ICE-only delay prediction with weather enrichment")
print("  • 6 JSON artifacts saved in data/reference/")
print("  • Merge: eva→coords + departure_planned_hour → hourly.time")
print("  • All notebooks load from disk — fully reproducible")
print()
print("What Comes Next:")
print("  → Notebook 02, Section 1: Load monthly Parquet, filter ICE, save to disk")
print(sep)

  [OK] Section 1 — project_config: project_config.json
  [OK] Section 2 — research_framework: research_framework.json
  [OK] Section 3 — db_data_dictionary: db_data_dictionary.json
  [OK] Section 4 — weather_data_dictionary: weather_data_dictionary.json
Section 5: Merge Strategy, Pipeline Design & Notebook Summary
Project: ICE Train Delay Prediction Using Railway Operational Data and Historical Weather Data

--- MERGE KEYS ---
  Railway station : eva → geocode → lat/lon
  Railway time      : departure_planned_time → floor to hour
  Weather location  : ['latitude', 'longitude']
  Weather time      : time (YYYY-MM-DDTHH:00 in Europe/Berlin)
  Join type         : left

--- PIPELINE (Notebooks 02–04) ---
  Step 1 [NB02]: Load monthly Parquet from Hugging Face
  Step 2 [NB02]: Filter train_type == 'ICE'
  Step 3 [NB03]: Standardize all timestamps to 24h datetime, timezone Europe/Berlin
  Step 4 [NB04]: Geocode unique eva station IDs to latitude/longitude
  Step 5 [NB04]: Fetch hourly weathe